# 3. Export real comparisons with provenance

Create CSV and JSON matching the viewer, plus one native real GHS-POP count cell as a supported rectangular GeoJSON import. The viewer also imports local count GeoTIFF windows using a cancellable browser worker; this standard-library notebook does not decode TIFF files. Keep source attribution, bounds, grid identity, coverage and method with every result.


In [ ]:
import json
from pathlib import Path
from population_tools_real import load_population, compare, validate_layer
data = load_population()
oracle = json.loads(Path('independent_oracle.json').read_text(encoding='utf-8'))
for layer in data['layers']:
    validate_layer(layer)
from population_tools_real import export_csv, METHOD, iter_cells, import_geojson
results = compare(data['layers'], data['default_aoi'])
for result in results:
    assert abs(result['population'] - oracle['sources'][result['metadata']['id']]['aoi_population']) < .0001
cell = next(c for c in iter_cells(data['layers'][3]) if c['population'] > 0)
w, s, e, n = cell['bounds']
metadata = dict(data['layers'][3]['metadata'], id='notebook-ghs-cell', title='GHS-POP native cell export', notebook_subset_bounds=cell['bounds'])
document = {'type':'FeatureCollection', 'metadata':metadata, 'features':[{'type':'Feature', 'geometry':{'type':'Polygon', 'coordinates':[[[w,s],[e,s],[e,n],[w,n],[w,s]]]}, 'properties':{'population':cell['population']}}]}
imported = import_geojson(document)
assert compare([imported], cell['bounds'])[0]['population'] == cell['population']
print('Validated real count cell:', cell['population'])


In [ ]:
output = Path('outputs')
output.mkdir(exist_ok=True)
(output / 'real-count-cell.geojson').write_text(json.dumps(document, indent=2), encoding='utf-8')
(output / 'real-comparison.json').write_text(json.dumps({'notice':'Real native source counts; CC BY 4.0; spatial subsets, no endorsement', 'bounds':data['default_aoi'], 'method':METHOD, 'results':results}, indent=2), encoding='utf-8')
(output / 'real-comparison.csv').write_text(export_csv(results), encoding='utf-8')
print('Created real-comparison.csv, real-comparison.json, real-count-cell.geojson')
print('REAL_EXPORT_BROWSER_OK')


Download files from the Lab `outputs` folder. JSON contains complete metadata and grid identity. CSV's `metadata_json` column preserves every supplied source notice and processing receipt. The viewer's SVG embeds the same data in its description and shows visible source credits. User-supplied provenance describes a declaration; structural validation does not independently prove an uploaded file's source identity.


## Sources and software

The complete citation, release, native geometry, source asset, NoData semantics, license, retrieval date (2026-11-06) and processing receipt travel in every layer's metadata. These are native-pixel Greater Lagos excerpts, spatially subset and losslessly serialized without resampling; no producer endorsement. All four products are CC BY 4.0. WorldPop: Bondarenko et al. (2025), https://doi.org/10.5258/SOTON/WP00839 (Global2 R2025A v1 alpha); GRID3: Nnanatu et al. (2025), https://doi.org/10.5258/SOTON/WP00782; LandScan: Rose et al. (2021), https://doi.org/10.48690/1523378; GHS-POP: Schiavina et al. (2023), https://doi.org/10.2905/2FF68A52-5B5B-4A22-8F40-C41DA8332CFE and Pesaresi et al. (2024), https://doi.org/10.1080/17538947.2024.2390454. Credit WorldPop / University of Southampton; GRID3 Inc. and CIESIN / Columbia Climate School; Oak Ridge National Laboratory; European Commission/JRC, (c) European Union, 1995-2026. Gates Foundation WorldPop INV-045237 and GRID3 INV-044979. Preserve all producer notices when sharing.

LandScan pixels came from a pinned Hugging Face mirror, not an ORNL-hosted TIFF. Independent ORNL statistics matched one cell and the complete prepared window; they do not prove full mirror bit identity. GHS-POP uses the official JRC source, not a quantized third-party mirror. GPW is not bundled; current Harvard Dataverse access uses a guestbook. GRID3 and WorldPop file servers currently lack browser CORS/range support; download yourself and use the viewer's bounded local TIFF import for other areas.

Inspired by NASA Earthdata, SEDAC/CIESIN and POPGRID Viewer (https://www.earthdata.nasa.gov/news/feature-articles/data-tool-focus-popgrid-viewer). Fitness for use: Leyk et al. (2019), https://doi.org/10.5194/essd-11-1385-2019. Execution: Project Jupyter / JupyterLite, Pyodide, Jupyter Book / MyST. The viewer uses Leaflet 1.9.4 and GeoTIFF.js 3.0.5 with their notices. RustGIS, GeoLibre, JupyterGIS and FieldWork integration remain future work.
